# SHL 2026 — OpenAI Whisper Large-v3 on two GPUs

Select **GPU T4 ×2** in Kaggle settings, then **Run All** yourself.
The competition data and `cruelangelssprint/openai-whisper-large-v3` must be attached.
Internet can remain off. OpenAI Whisper Large-v3 runs locally on each GPU;
there are no API calls or API keys.

One independent worker runs on each GPU. Recordings are balanced by audio duration,
transcribed in batches, and checkpointed after each batch. Live output shows each
GPU's progress, processing speed, error count, and ETA. Rerunning in the same session
reuses successful checkpoints and retries failures.


## Controlled transcription comparison

This is a separate run for **Whisper Large-v3**, to compare with the saved Whisper
Small transcripts. The same English transcription task, 30-second chunking, audio
resampling, and two-GPU worker strategy are retained. Batch size is reduced from
16 to 4 for GPU memory. Checkpoints and final filenames are separated from Small.

Download **transcripts_large_v3.parquet** and **transcripts_large_v3.csv** after
all 985 recordings finish with zero errors. The scoring experiment will reuse the
original Ridge/TF-IDF configuration and all six original folds. Improved recognition
does not guarantee improved grammar-score RMSE.


In [ ]:
from pathlib import Path
from IPython.display import Audio, FileLink, display
import importlib.util
import json
import pandas as pd
import soundfile as sf
import torch

required = ['transformers', 'scipy', 'pyarrow', 'accelerate']
missing = [name for name in required if importlib.util.find_spec(name) is None]
assert not missing, f'Missing dependencies in the Kaggle image: {missing}'
assert torch.cuda.is_available(), 'Select GPU T4 ×2, then start a new session.'
assert torch.cuda.device_count() == 2, 'This notebook requires two GPUs: select GPU T4 ×2.'
for gpu in range(2):
    print(f'GPU {gpu}: {torch.cuda.get_device_name(gpu)}')

BATCH_SIZE = 4  # Reduce to 8 if necessary; failed batches also retry at smaller sizes.
WORKING = Path('/kaggle/working/whisper_large_v3')
WORKING.mkdir(parents=True, exist_ok=True)
INPUT_ROOT = Path('/kaggle/input')
train_candidates = [p for p in INPUT_ROOT.rglob('train.csv')
                    if 'shl-hiring-assessment-2026' in str(p)]
assert len(train_candidates) == 1, 'Attach the SHL Hiring Assessment 2026 competition.'
DATA_ROOT = train_candidates[0].parent
train_csv = DATA_ROOT / 'train.csv'
test_csv = DATA_ROOT / 'test.csv'
train_table, test_table = pd.read_csv(train_csv), pd.read_csv(test_csv)
rows = []
for split, table in [('train', train_table), ('test', test_table)]:
    files = sorted((DATA_ROOT / split).rglob('*.wav'))
    assert len(files) == len(table), f'{split}: {len(files)} WAV files but {len(table)} CSV rows'
    for path in files:
        info = sf.info(path)
        rows.append({'audio_id': path.name, 'split': split,
                     'audio_path': str(path), 'duration_s': info.duration})
jobs = pd.DataFrame(rows)
assert jobs['split'].value_counts().to_dict() == {'train': 769, 'test': 216}
assert not jobs.duplicated(['split', 'audio_id']).any()
print(f'{len(jobs):,} recordings | {jobs.duration_s.sum()/3600:.2f} hours of audio')
display(jobs.groupby('split').agg(files=('audio_id', 'size'), audio_minutes=('duration_s', lambda s: s.sum()/60)))
print('Sample recording:')
display(Audio(jobs.iloc[0].audio_path))

# Kaggle supports both the legacy and the newer /datasets/ input mount layout.
model_candidates = []
for config_path in INPUT_ROOT.rglob('config.json'):
    if 'openai-whisper-large-v3' not in str(config_path):
        continue
    config = json.loads(config_path.read_text())
    if config.get('model_type') == 'whisper' and config.get('d_model') == 1280 and config.get('encoder_layers') == 32 and config.get('decoder_layers') == 32 and config.get('num_mel_bins') == 128:
        model_candidates.append(config_path.parent)
assert len(model_candidates) == 1, 'Attach cruelangelssprint/openai-whisper-large-v3 with Hugging Face Whisper Large-v3 weights.'
MODEL_DIR = model_candidates[0]
print(f'OpenAI Whisper Large-v3 weights: {MODEL_DIR}')

processor_config = json.loads((MODEL_DIR / 'preprocessor_config.json').read_text())
assert processor_config.get('feature_size') == 128, 'Large-v3 requires its matching 128-bin audio processor.'
assert (MODEL_DIR / 'model.safetensors').exists(), 'Expected the attached Large-v3 safetensors weights.'


In [ ]:
WORKER_SCRIPT = WORKING / "whisper_batch_worker.py"
worker_source = "import gc\nimport json\nimport math\nimport sys\nimport time\nfrom pathlib import Path\n\nimport numpy as np\nimport pandas as pd\nimport soundfile as sf\nimport torch\nfrom scipy.signal import resample_poly\nfrom transformers import AutoModelForSpeechSeq2Seq, AutoProcessor, pipeline\n\njob_file, output_file, model_dir, worker_name, batch_size = sys.argv[1:6]\noutput_file = Path(output_file)\nbatch_size = int(batch_size)\njobs = pd.read_parquet(job_file)\nassert torch.cuda.device_count() == 1, 'Each worker must see only its assigned GPU.'\ntorch.set_num_threads(2)\nmodel = AutoModelForSpeechSeq2Seq.from_pretrained(\n    model_dir, torch_dtype=torch.float16, low_cpu_mem_usage=True,\n    local_files_only=True, use_safetensors=True,\n).to('cuda:0')\nprocessor = AutoProcessor.from_pretrained(model_dir, local_files_only=True)\nasr = pipeline(\n    'automatic-speech-recognition', model=model,\n    tokenizer=processor.tokenizer, feature_extractor=processor.feature_extractor,\n    torch_dtype=torch.float16, device=0, chunk_length_s=30, batch_size=batch_size,\n)\n\nfinished = []\nif output_file.exists():\n    previous = pd.read_parquet(output_file)\n    assert 'model_id' in previous and previous['model_id'].eq('openai/whisper-large-v3').all(), 'Checkpoint belongs to another model; use a fresh Large-v3 output folder.'\n    successful = previous['error'].isna() & previous['transcript'].fillna('').str.strip().ne('')\n    # Reuse only matching recordings from this worker's current partition.\n    expected = set(zip(jobs['split'], jobs['audio_id']))\n    finished = [r for r in previous.loc[successful].to_dict('records')\n                if (r['split'], r['audio_id']) in expected]\ncompleted = {(r['split'], r['audio_id']) for r in finished}\nremaining = [r for r in jobs.to_dict('records') if (r['split'], r['audio_id']) not in completed]\nprint(f'[{worker_name}] {torch.cuda.get_device_name(0)} | resuming {len(finished)} | remaining {len(remaining)}', flush=True)\n\n\ndef load_audio(path):\n    audio, rate = sf.read(path, dtype='float32', always_2d=True)\n    audio = audio.mean(axis=1)\n    if rate != 16000:\n        divisor = math.gcd(rate, 16000)\n        audio = resample_poly(audio, 16000 // divisor, rate // divisor).astype(np.float32)\n    return {'raw': audio, 'sampling_rate': 16000}\n\n\ndef transcribe(rows):\n    try:\n        inputs = [load_audio(row['audio_path']) for row in rows]\n        results = asr(inputs, batch_size=min(batch_size, len(rows)),\n                      generate_kwargs={'language': 'english', 'task': 'transcribe'})\n        if len(results) != len(rows):\n            raise RuntimeError('Whisper returned an unexpected number of results.')\n        output = []\n        for row, result in zip(rows, results):\n            transcript = result['text'].strip()\n            output.append({**row, 'transcript': transcript,\n                           'error': None if transcript else 'Whisper returned empty text',\n                           'model_id': 'openai/whisper-large-v3'})\n        return output\n    except Exception as exc:\n        error = str(exc)\n    # Retry smaller batches on OOM or decoding errors; isolate bad recordings.\n    gc.collect()\n    torch.cuda.empty_cache()\n    if len(rows) > 1:\n        middle = len(rows) // 2\n        print(f'[{worker_name}] retrying batch of {len(rows)} in smaller batches: {error[:180]}', flush=True)\n        return transcribe(rows[:middle]) + transcribe(rows[middle:])\n    return [{**rows[0], 'transcript': None, 'error': error,\n             'model_id': 'openai/whisper-large-v3'}]\n\n\ndef checkpoint():\n    temporary = output_file.with_suffix('.tmp.parquet')\n    pd.DataFrame(finished).to_parquet(temporary, index=False)\n    temporary.replace(output_file)\n\n\nstarted = time.perf_counter()\nprocessed_seconds = 0.0\nremaining_seconds = sum(row['duration_s'] for row in remaining)\nfor start in range(0, len(remaining), batch_size):\n    batch = remaining[start:start + batch_size]\n    finished.extend(transcribe(batch))\n    checkpoint()\n    processed_seconds += sum(row['duration_s'] for row in batch)\n    elapsed = time.perf_counter() - started\n    rate = processed_seconds / max(elapsed, 0.001)\n    eta = (remaining_seconds - processed_seconds) / max(rate, 0.001)\n    errors = sum(row['error'] is not None for row in finished)\n    print(f'[{worker_name}] {len(finished)}/{len(jobs)} files | '\n          f'{rate:.1f}x audio speed | elapsed {elapsed/60:.1f} min | '\n          f'ETA {eta/60:.1f} min | errors {errors}', flush=True)\ncheckpoint()\nprint(f'[{worker_name}] complete', flush=True)\n"
WORKER_SCRIPT.write_text(worker_source)
print(f"Worker saved to {WORKER_SCRIPT}")

In [ ]:
import os
import subprocess
import sys
import time

# Balance total recording duration across the two GPUs.
assignments = [[], []]
loads = [0.0, 0.0]
for row in jobs.sort_values('duration_s', ascending=False).to_dict('records'):
    gpu = min(range(2), key=lambda i: loads[i])
    assignments[gpu].append(row)
    loads[gpu] += row['duration_s']
job_files = [WORKING / f'jobs_gpu_{i}.parquet' for i in range(2)]
output_files = [WORKING / f'transcripts_gpu_{i}.parquet' for i in range(2)]
for gpu in range(2):
    pd.DataFrame(assignments[gpu]).to_parquet(job_files[gpu], index=False)
    print(f'GPU {gpu}: {len(assignments[gpu])} recordings | {loads[gpu]/60:.1f} audio minutes')

processes = []
try:
    for gpu in range(2):
        environment = {**os.environ, 'CUDA_VISIBLE_DEVICES': str(gpu),
                       'TOKENIZERS_PARALLELISM': 'false', 'OMP_NUM_THREADS': '2',
                       'HF_HUB_OFFLINE': '1', 'TRANSFORMERS_OFFLINE': '1'}
        command = [sys.executable, '-u', str(WORKER_SCRIPT), str(job_files[gpu]),
                   str(output_files[gpu]), str(MODEL_DIR), f'GPU {gpu}', str(BATCH_SIZE)]
        # Workers inherit notebook stdout/stderr so progress appears below this cell.
        processes.append(subprocess.Popen(command, env=environment))
    while any(process.poll() is None for process in processes):
        failures = [process.returncode for process in processes
                    if process.poll() is not None and process.returncode != 0]
        if failures:
            raise RuntimeError(f'GPU worker failed: {failures}. See output above.')
        time.sleep(1)
    assert all(process.returncode == 0 for process in processes), 'A GPU worker failed.'
finally:
    # Interrupting this notebook cell also stops both GPU workers.
    for process in processes:
        if process.poll() is None:
            process.terminate()
    for process in processes:
        try:
            process.wait(timeout=10)
        except subprocess.TimeoutExpired:
            process.kill()
            process.wait()
print('Both GPU workers completed.')


In [ ]:
transcripts = pd.concat([pd.read_parquet(path) for path in output_files], ignore_index=True)
transcripts = transcripts.sort_values(['split', 'audio_id']).reset_index(drop=True)
assert len(transcripts) == len(jobs), 'Missing transcription rows.'
assert not transcripts.duplicated(['split', 'audio_id']).any()
assert set(zip(transcripts['split'], transcripts['audio_id'])) == set(zip(jobs['split'], jobs['audio_id']))

# Keep audio_id as the original WAV filename so later training notebooks can join it.
# audio_path is useful only in this session; audio_id and split are portable.
FINAL_PARQUET = Path('/kaggle/working/transcripts_large_v3.parquet')
FINAL_CSV = Path('/kaggle/working/transcripts_large_v3.csv')
transcripts.to_parquet(FINAL_PARQUET, index=False)
transcripts.to_csv(FINAL_CSV, index=False)
for split in ['train', 'test']:
    transcripts.loc[transcripts['split'].eq(split)].to_csv(Path('/kaggle/working') / f'{split}_transcripts_large_v3.csv', index=False)

failed = transcripts.error.notna() | transcripts.transcript.fillna('').str.strip().eq('')
manifest = {'model_id': 'openai/whisper-large-v3', 'language': 'english',
            'chunk_length_s': 30, 'sampling_rate': 16000, 'task': 'transcribe', 'gpu_count': 2, 'batch_size': BATCH_SIZE, 'total': len(transcripts),
            'split_counts': transcripts['split'].value_counts().to_dict(),
            'errors': int(failed.sum())}
(Path('/kaggle/working/transcription_manifest_large_v3.json')).write_text(json.dumps(manifest, indent=2))
assert transcripts.model_id.eq('openai/whisper-large-v3').all()
print(f'Saved {len(transcripts):,} transcripts | errors: {failed.sum()}')
display(transcripts[['audio_id', 'split', 'transcript', 'error']].head(10))
for path in [FINAL_PARQUET, FINAL_CSV, Path('/kaggle/working/train_transcripts_large_v3.csv'), Path('/kaggle/working/test_transcripts_large_v3.csv')]:
    display(FileLink(str(path)))
if (~failed).any():
    sample = transcripts.loc[~failed].iloc[0]
    print(f'Sample file: {sample.audio_id}\nTranscript: {sample.transcript}')
    display(Audio(sample.audio_path))
if failed.any():
    display(transcripts.loc[failed, ['audio_id', 'split', 'error']])
    raise RuntimeError('Outputs saved with errors. Rerun the worker cell and this cell to retry failed recordings before reusing the dataset.')


## Save once and import in competition training notebooks

After the run succeeds, save the notebook version **with its outputs**, or download
`transcripts_large_v3.parquet` from the output links above. Create a **private Kaggle Dataset**
containing that file and attach it to your later training notebooks.

```python
from pathlib import Path
import pandas as pd

# Replace this with the actual mount path shown in your attached dataset.
transcripts_path = Path('/kaggle/input/YOUR-TRANSCRIPTS-DATASET/transcripts_large_v3.parquet')
transcripts = pd.read_parquet(transcripts_path)
train_text = transcripts.loc[transcripts['split'].eq('train'), ['audio_id', 'transcript']]
test_text = transcripts.loc[transcripts['split'].eq('test'), ['audio_id', 'transcript']]

# If the competition CSV uses `filename` as its audio identifier:
train = train.merge(train_text, left_on='filename', right_on='audio_id',
                    how='left', validate='one_to_one')
test = test.merge(test_text, left_on='filename', right_on='audio_id',
                  how='left', validate='one_to_one')
assert train.transcript.notna().all() and test.transcript.notna().all()
```

These files contain transcription features for your scoring model. Train your grammar
scoring model on the labelled competition data and generate its submission separately.
